###### Setup and Module Imports

In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath("../src"))
from ab_testing import check_srm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Environment configured successfully.")

Environment configured successfully.


###### Raw Data Ingestion & Sanity Check

In [2]:
# Load dataset from data/raw/
raw_data_path = "../data/raw/ab_data.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Total Raw Records Loaded: {len(df_raw):,}")
print("\nFirst 5 Records:")
display(df_raw.head())

print("\nData Types and Missing Values:")
print(df_raw.info())

Total Raw Records Loaded: 294,478

First 5 Records:


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1



Data Types and Missing Values:
<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   user_id       294478 non-null  int64
 1   timestamp     294478 non-null  str  
 2   group         294478 non-null  str  
 3   landing_page  294478 non-null  str  
 4   converted     294478 non-null  int64
dtypes: int64(2), str(3)
memory usage: 23.0 MB
None


###### Identify Mismatched Group & Page Assignments

In [3]:
# Crosstabulate group vs landing_page
crosstab_raw = pd.crosstab(df_raw['group'], df_raw['landing_page'], margins=True)
print("--- Raw Group vs. Landing Page Matrix ---")
print(crosstab_raw)

# Identify mismatched rows
mismatch_mask = ((df_raw['group'] == 'treatment') ^ (df_raw['landing_page'] == 'new_page'))
num_mismatches = mismatch_mask.sum()

print(f"\nMismatched Records Found: {num_mismatches:,} ({num_mismatches / len(df_raw):.2%} of total data)")

--- Raw Group vs. Landing Page Matrix ---
landing_page  new_page  old_page     All
group                                   
control           1928    145274  147202
treatment       145311      1965  147276
All             147239    147239  294478

Mismatched Records Found: 3,893 (1.32% of total data)


###### Clean Data & Deduplicate Users

In [4]:
# Retain only valid group-page matches
df_aligned = df_raw[~mismatch_mask].copy()

# Identify duplicate user IDs
duplicate_count = df_aligned.duplicated(subset=['user_id']).sum()
print(f"Duplicate User IDs Found: {duplicate_count:,}")

# Deduplicate user IDs (keep initial log)
df_clean = df_aligned.drop_duplicates(subset=['user_id'], keep='first').copy()

# Format timestamp and construct date dimensions
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean['date'] = df_clean['timestamp'].dt.date

print(f"\nFinal Cleaned Unique Users: {len(df_clean):,}")
print(f"Control Users (A):   {len(df_clean[df_clean['group'] == 'control']):,}")
print(f"Treatment Users (B): {len(df_clean[df_clean['group'] == 'treatment']):,}")

Duplicate User IDs Found: 1

Final Cleaned Unique Users: 290,584
Control Users (A):   145,274
Treatment Users (B): 145,310


###### Sample Ratio Mismatch (SRM) Validation

In [5]:
# Get observed group counts
group_counts = df_clean['group'].value_counts()

# Run Chi-Square test using src/ab_testing.py
srm_results = check_srm(
    control_count=group_counts['control'], 
    treatment_count=group_counts['treatment'], 
    expected_ratio=0.50
)

print("--- Sample Ratio Mismatch (SRM) Check ---")
for key, value in srm_results.items():
    print(f"{key}: {value}")

--- Sample Ratio Mismatch (SRM) Check ---
control_count: 145274
treatment_count: 145310
chi2_stat: 0.004459984032155934
p_value: 0.9467543681597944
has_srm: False
status: PASS (No SRM)


###### Export Processed Datasets for Modeling and Looker Studio

In [6]:
# Create processed directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# 1. Export cleaned microdata
df_clean.to_csv("../data/processed/ab_data_cleaned.csv", index=False)

# 2. Export daily aggregated metrics for Looker Studio
df_daily = df_clean.groupby(['date', 'group', 'landing_page']).agg(
    total_users=('user_id', 'count'),
    total_conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean')
).reset_index()

df_daily.to_csv("../data/processed/ab_daily_summary.csv", index=False)

print("Successfully exported datasets to data/processed/:")
print(" - ab_data_cleaned.csv")
print(" - ab_daily_summary.csv")

Successfully exported datasets to data/processed/:
 - ab_data_cleaned.csv
 - ab_daily_summary.csv
